# BT4012 Kaggle Competition 

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = "dataset/"        # folder with train.csv, test.csv, txs_edgelist.csv
SEED = 42                    # fixed seed for reproducibility
SEEDS = [0, 1, 2, 3, 4]      # the final model is trained 5 times with these seeds and averaged
N_DRIFT_DROP = 20            # number of most-drifting features to remove
RECENCY_SCALE = 15           # weight of decay rate

np.random.seed(SEED)

## 1. Load Data

- **`train.csv`** — `label` column is `1` (illicit), `0` (licit) or unknown.
- **`test.csv`** 
- **`txs_edgelist.csv`** — links between transactions: `txId1 → txId2` means `txId2` spent money created by `txId1`.

Unknown labels are turned into missing values (`NaN`). Ignored for training

In [2]:
train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")
edges = pd.read_csv(DATA_DIR + "txs_edgelist.csv")

# 1 = illicit, 0 = licit, anything else ("unknown", blank) -> NaN
train["label"] = pd.to_numeric(train["label"], errors="coerce")

# The 165 anonymised transaction features
FEATS = [c for c in train.columns if c.startswith("feat_")]

print(f"train: {train.shape}   test: {test.shape}   edges: {edges.shape}   features: {len(FEATS)}")
print(train["label"].value_counts(dropna=False).rename({0.0: "licit (0)", 1.0: "illicit (1)"}))

train: (141772, 168)   test: (15329, 168)   edges: (234355, 2)   features: 165
label
NaN            110537
licit (0)       27591
illicit (1)      3644
Name: count, dtype: int64


## 2. Explore the data

In [5]:
print("train time steps:", train.time_step.min(), "-", train.time_step.max())
print("test  time steps:", test.time_step.min(),  "-", test.time_step.max())

# labelled transactions and illicit rate per time step
by_step = train.groupby("time_step")["label"].agg(n_labelled="count", illicit_rate="mean")
display(by_step.T.round(3))

# edges whose endpoints are missing from both files
step_of = pd.concat([train[["txId", "time_step"]], test[["txId", "time_step"]]]).set_index("txId")["time_step"]
both_known = edges.txId1.isin(step_of.index) & edges.txId2.isin(step_of.index)
n_missing = (~edges.txId1.isin(step_of.index)).sum() + (~edges.txId2.isin(step_of.index)).sum()
print(f"\nedge endpoints missing from both files: {n_missing:,}")

#do edges with both ends known stay within same time step
same_step = (edges[both_known].txId1.map(step_of) == edges[both_known].txId2.map(step_of)).mean()
print(f"share of (complete) edges within the same time step: {same_step:.3f}")

#original dataset has 46564 labelled transactions in total
n_lab_train = train.label.notna().sum()
print(f"\n46,564 labelled in Elliptic - {n_lab_train:,} labelled in train = {46564 - n_lab_train:,}"
      f"   (test set size: {len(test):,})")

train time steps: 1 - 35
test  time steps: 36 - 49


time_step,1,2,3,4,5,6,7,8,9,10,...,26,27,28,29,30,31,32,33,34,35
n_labelled,2147.000,1117.000,1279.000,1440.000,1882.000,485.00,1203.000,1165.000,778.000,972.000,...,517.000,206.000,284.000,1174.00,524.000,710.000,1323.000,441.000,515.000,1341.000
illicit_rate,0.008,0.016,0.009,0.021,0.004,0.01,0.085,0.058,0.319,0.019,...,0.186,0.117,0.299,0.28,0.158,0.149,0.259,0.052,0.072,0.136



edge endpoints missing from both files: 92,629
share of (complete) edges within the same time step: 1.000

46,564 labelled in Elliptic - 31,235 labelled in train = 15,329   (test set size: 15,329)


**Findings:** Test size matches exactly the labelled transactions left over from the original dataset. So the test set is only labelled transactions with the unlabelled neighbours removed. Accounted for in the graph features in the next section.

## 3. Graph features

The original features mostly describe a transaction on its own. But illicit money is often revealed by where it comes from and where it goes next. Each transaction also gets features describing its neighbours:

| Feature | What it captures |
|---|---|
| In-degree / out-degree | How many transactions send money into it, and how many it pays into. Unusual values can signal mixing or splitting of funds. |
| Average features of predecessors | What the transactions sending money *in* typically look like. |
| Average features of successors | What the transactions receiving money *out* typically look like. |
| Difference from neighbourhood average | How much the transaction stands out from those around it. |

**Important fix — only labelled transactions are used.** Training transactions still have their unlabelled neighbours, but test transactions lost theirs. Built on the full graph, test transactions would look more isolated than they really are. So the graph is built using only edges between labelled transactions, which makes training and test data look the same. 

The averages are computed efficiently with sparse matrix multiplication.

In [6]:
# make a table with every transaction (train + test), and a lookup from txId -> row number
allx = pd.concat([train.drop(columns="label"), test], ignore_index=True)
if "index" in allx.columns:
    allx = allx.drop(columns="index")
id2row = pd.Series(np.arange(len(allx)), index=allx["txId"])
n = len(allx)

def row_normalise(M):
    # Divide each row by its sum, so that (M @ X) gives an average over neighbours
    d = np.asarray(M.sum(1)).ravel()
    d[d == 0] = 1
    return sp.diags(1 / d) @ M

# Keep only edges where BOTH ends are labelled transactions (labelled train rows + all test rows)
keep_ids = set(train.loc[train.label.notna(), "txId"]) | set(test["txId"])
e = edges[edges.txId1.isin(keep_ids) & edges.txId2.isin(keep_ids)]
src, dst = id2row[e.txId1].values, id2row[e.txId2].values

# Adjacency matrix: AL[i, j] = 1 means transaction i pays into transaction j
AL = sp.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(n, n))

X_all     = allx[FEATS].values.astype(np.float32)
out_deg   = np.asarray(AL.sum(1)).ravel()          # number of transactions it pays into
in_deg    = np.asarray(AL.sum(0)).ravel()          # number of transactions paying into it
succ_mean = row_normalise(AL) @ X_all              # average features of successors
pred_mean = row_normalise(AL.T) @ X_all            # average features of predecessors

graph_feats = pd.concat([
    pd.DataFrame({"L_in_deg": in_deg, "L_out_deg": out_deg, "L_log_total_deg": np.log1p(in_deg + out_deg)}),
    pd.DataFrame(pred_mean, columns=[f"L_pred_mean_{c}" for c in FEATS]),
    pd.DataFrame(succ_mean, columns=[f"L_succ_mean_{c}" for c in FEATS]),
    pd.DataFrame(X_all - (pred_mean + succ_mean) / 2, columns=[f"L_diff_{c}" for c in FEATS]),
], axis=1)
graph_feats["txId"] = allx["txId"].values
GRAPH_FEATS = [c for c in graph_feats.columns if c != "txId"]

# Attach graph features. `lab` = labelled training rows only; `test_g` = test rows.
lab    = train[train.label.notna()].reset_index(drop=True).merge(graph_feats, on="txId", how="left")
test_g = test.merge(graph_feats, on="txId", how="left")
BASE   = FEATS + GRAPH_FEATS   # 165 raw + 498 graph features

print(f"edges kept (labelled-only graph): {len(e):,} of {len(edges):,}")
print(f"graph features: {len(GRAPH_FEATS)}   labelled training rows: {len(lab):,}   illicit share: {lab.label.mean():.4f}")

edges kept (labelled-only graph): 36,624 of 234,355
graph features: 498   labelled training rows: 31,235   illicit share: 0.1167


## 4. Validation setup and model comparison

**Forward-in-time validation.** Because the test period comes after training, validation must also go forward in time: train on earlier steps, validate on the next ones. A random split would let the model "see the future" and give over-optimistic scores.

Three folds are used: train on steps 1–26 / 1–29 / 1–32 and validate on the following 3 steps.

**Class imbalance.** AUC only looks at ranking, so no decision threshold is needed. Illicit transactions are given more weight in training (`scale_pos_weight` = licit ÷ illicit) so the model does not ignore the rarer class.


In [7]:
steps = np.sort(lab.time_step.unique())
n_folds = 3
val_block = max(1, int(len(steps) * 0.3) // n_folds)
folds = []
for k in range(n_folds):
    v_end = len(steps) - (n_folds - 1 - k) * val_block
    v_start = v_end - val_block
    folds.append((steps[:v_start], steps[v_start:v_end]))
for tr_steps, va_steps in folds:
    print(f"train on steps {tr_steps.min()}-{tr_steps.max()}  ->  validate on steps {va_steps.min()}-{va_steps.max()}")

def temporal_cv(make_model, cols):
    # train on each fold's past, score its future; return the mean AUC
    aucs = []
    for tr_steps, va_steps in folds:
        tr, va = lab[lab.time_step.isin(tr_steps)], lab[lab.time_step.isin(va_steps)]
        m = make_model()
        m.fit(tr[cols], tr.label)
        aucs.append(roc_auc_score(va.label, m.predict_proba(va[cols])[:, 1]))
    return round(float(np.mean(aucs)), 4)

pos_w = (lab.label == 0).sum() / (lab.label == 1).sum()   # licit / illicit ratio, used as class weight
print(f"\nclass weight for illicit (scale_pos_weight): {pos_w:.2f}")

train on steps 1-26  ->  validate on steps 27-29
train on steps 1-29  ->  validate on steps 30-32
train on steps 1-32  ->  validate on steps 33-35

class weight for illicit (scale_pos_weight): 7.57


In [8]:
def lgbm(seed=SEED):
    # standard LightGBM: used for model comparison and inside prediction propagation
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              scale_pos_weight=pos_w, random_state=seed, verbose=-1)

def lgbm_reg(seed=SEED):
    # regularised LightGBM for the final model: slower learning, smaller trees, fewer features per tree.
    # relies on many features rather than a few, so it copes better when the data changes over time.
    return lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.01, num_leaves=15, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.3, reg_lambda=5.0,
                              scale_pos_weight=pos_w, random_state=seed, verbose=-1)

# compare model types
candidates = {
    "Logistic regression (raw features)": (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")), FEATS),
    "Random forest (raw features)":       (lambda: RandomForestClassifier(n_estimators=300, n_jobs=-1, class_weight="balanced_subsample", random_state=SEED), FEATS),
    "LightGBM (raw features)":            (lgbm, FEATS),
    "LightGBM (raw + graph features)":    (lgbm, BASE),
}
comparison = pd.Series({name: temporal_cv(mk, cols) for name, (mk, cols) in candidates.items()}, name="mean validation AUC")
display(comparison.to_frame())

,mean validation AUC
Logistic regression (raw features),0.9377
Random forest (raw features),0.9966
LightGBM (raw features),0.9978
LightGBM (raw + graph features),0.9977


## 5. Drift check 

Validation scores were far higher than leaderboard scores, which suggests the test period looks different from training. Test:

1. Take the last 10 training time steps (labelled **0**) and the test set (labelled **1**).
2. Train a classifier to tell them apart from the features alone.
3. If the periods were similar, it could only guess . The higher the AUC, the bigger the change.

The features the classifier relies on most are the ones that changed most. A model trained on their old values may be misled by their new values, so the 20 most changed features are removed from the final model. Removing more would also throw away useful signal.

In [11]:
recent = lab[lab.time_step >= lab.time_step.max() - 9]          # last 10 training steps
adv = pd.concat([recent[BASE].assign(is_test=0), test_g[BASE].assign(is_test=1)], ignore_index=True)

# how well can the two periods be told apart? 
adv_oof = np.zeros(len(adv))
for tr_i, va_i in GroupKFold(n_splits=5).split(adv, groups=np.arange(len(adv)) % 5):
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, colsample_bytree=0.5, verbose=-1, random_state=SEED)
    m.fit(adv.iloc[tr_i][BASE], adv.is_test.iloc[tr_i])
    adv_oof[va_i] = m.predict_proba(adv.iloc[va_i][BASE])[:, 1]
print(f"adversarial AUC (0.5 = no change, 1.0 = completely different): {roc_auc_score(adv.is_test, adv_oof):.4f}")

# which features changed most? 
m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, colsample_bytree=0.5, verbose=-1,
                       random_state=SEED, importance_type="gain").fit(adv[BASE], adv.is_test)
drift = pd.Series(m.feature_importances_, index=BASE).sort_values(ascending=False)
print("\nmost changed features:")
print(drift.head(10).round(1).to_string())

DROP = list(drift.index[:N_DRIFT_DROP])
print(f"\nremoving the top {N_DRIFT_DROP} drifting features")

adversarial AUC (0.5 = no change, 1.0 = completely different): 1.0000

most changed features:
feat_101                172209.4
feat_103                 52728.0
feat_136                 24817.6
feat_100                 17188.7
L_diff_feat_103           6841.2
L_succ_mean_feat_103      5660.4
L_pred_mean_feat_103      1908.7
feat_139                  1082.6
L_succ_mean_feat_100       803.5
L_diff_feat_136            743.4

removing the top 20 drifting features


## 6. Prediction propagation

Illicit transactions tend to be connected: money from one scam often flows into other suspicious transactions. So knowing that a transaction's neighbour is risky is a strong clue. This is done in two steps:

1. Score every transaction with LightGBM.
2. Add the neighbours' scores as new featuresm the average and highest score of its predecessors and of its successors, and the average score two steps away — then retrain.

**Avoiding leakage.** A training transaction must not be scored by a model that has seen its own label, or its neighbours' labels. So training scores are out-of-fold, the data is split into 5 groups by time step, and each group is scored by a model trained on the other four. Transactions only link to others in the same time step, so a transaction and all its neighbours are always in the same group.

This is repeated twice, the second round propagates scores from a model that already uses the first round's features, so risk spreads further through the graph. Prediction propagation was the single largest improvement 

In [12]:
A_und = ((AL + AL.T) > 0).astype(np.float32).tocsr()   # same graph, ignoring direction 

def pp_features(score, tag):
    # 1 risk score per transaction into neighbour features:
    # mean / max score of predecessors and successors, plus the mean score two steps away.
    has = (~np.isnan(score)).astype(np.float32)        # 1 where a score exists
    s0  = np.nan_to_num(score)
    out = {}
    for M, direction in [(AL.T.tocsr(), "pred"), (AL.tocsr(), "succ")]:
        cnt  = M @ has
        mean = (M @ s0) / np.maximum(cnt, 1)
        mx   = np.asarray(M.multiply(s0[None, :]).tocsr().max(axis=1).todense()).ravel()
        mean[cnt == 0] = np.nan                         # no neighbours -> missing (LightGBM handles NaN)
        mx[cnt == 0] = np.nan
        out[f"{tag}_{direction}_mean"], out[f"{tag}_{direction}_max"] = mean, mx
    cnt1 = A_und @ has
    hop1 = np.where(cnt1 > 0, (A_und @ s0) / np.maximum(cnt1, 1), 0).astype(np.float32)
    cnt2 = A_und @ (cnt1 > 0).astype(np.float32)
    hop2 = (A_und @ hop1) / np.maximum(cnt2, 1)
    hop2[cnt2 == 0] = np.nan
    out[f"{tag}_2hop_mean"] = hop2
    F = pd.DataFrame(out)
    F["txId"] = allx["txId"].values
    return F

def oof_scores(tr, te, cols):
    # out-of-fold scores for training rows + scores for test rows
    oof = np.zeros(len(tr))
    for tr_i, va_i in GroupKFold(n_splits=5).split(tr, groups=tr.time_step):
        m = lgbm(SEED)
        m.fit(tr.iloc[tr_i][cols], tr.label.iloc[tr_i])
        oof[va_i] = m.predict_proba(tr.iloc[va_i][cols])[:, 1]
    m = lgbm(SEED)
    m.fit(tr[cols], tr.label)
    print(f"  out-of-fold AUC: {roc_auc_score(tr.label, oof):.4f}")
    score = np.full(n, np.nan, dtype=np.float32)
    score[id2row[tr.txId].values] = oof
    score[id2row[te.txId].values] = m.predict_proba(te[cols])[:, 1]
    return score

def attach(F):
    # add new neighbour features to the training and test tables
    cols = [c for c in F.columns if c != "txId"]
    return lab.merge(F, on="txId", how="left"), test_g.merge(F, on="txId", how="left"), cols

print("round 1")
score1 = oof_scores(lab, test_g, BASE)
tr1, te1, C1 = attach(pp_features(score1, "it1"))

print("round 2")
score2 = oof_scores(tr1, te1, BASE + C1)
tr2, te2, C2 = attach(pp_features(score2, "it2"))
print(f"\npropagation features added: {C2}")

round 1
  out-of-fold AUC: 0.9982
round 2
  out-of-fold AUC: 0.9981

propagation features added: ['it2_pred_mean', 'it2_pred_max', 'it2_succ_mean', 'it2_succ_max', 'it2_2hop_mean']


## 7. Final model and submission

The final model combines everything that helped on the leaderboard:

- **Features:** raw + graph features, minus the 20 most drifting ones, plus the round-2 propagation features.
- **Regularised LightGBM** (`lgbm_reg`, defined in Section 4).
- **Recency weighting:** each training row is weighted by $e^{(t-35)/15}$, so the latest step (35) counts fully and older steps count less, down to about 10% for step 1. The value 15 was chosen by judgement, not tuned.
- **Seed averaging:** trained 5 times with different random seeds; the predictions are converted to ranks and averaged, which makes them steadier. (10 seeds did not do better.)

In [14]:
def rank01(p):
    # convert scores to ranks between 0 and 1, so runs on slightly different scales average fairly
    return (rankdata(p) - 1) / (len(p) - 1)

def seed_average(cols, weight, make_model, tr, te):
    # train the same model once per seed and average the ranked test predictions
    preds = []
    for s in SEEDS:
        m = make_model(s)
        m.fit(tr[cols], tr.label, sample_weight=weight)
        preds.append(rank01(m.predict_proba(te[cols])[:, 1]))
    return np.mean(preds, axis=0)

FINAL_COLS = [c for c in BASE if c not in DROP] + C2
recency_w  = np.exp((tr2.time_step - tr2.time_step.max()) / RECENCY_SCALE).values   # step 35 -> 1.0, step 1 -> ~0.10

final_pred = seed_average(FINAL_COLS, recency_w, lgbm_reg, tr2, te2)

idx = test["index"].values if "index" in test.columns else np.arange(len(test))
submission = pd.DataFrame({"index": idx, "target": final_pred})
submission.to_csv("submission.csv", index=False)
print(f"final features used: {len(FINAL_COLS)}")

final features used: 648
